# Emergency Contacts Manager: Final Project (Linked List)

**Course:** Data Structures  |  **Language:** Python

This notebook is the final result of the project. It manages emergency contacts using a **singly linked list** and three hand-written algorithms: **Linear Search**, **Bubble Sort** and **Selection Sort**.

Each contact has: **Contact ID, Name, Phone Number, Category, Priority** (1 = highest priority).

| Section | What it covers | Marks |
|---|---|---|
| 1 | Linked list implementation | 4 |
| 2 | Add, display, update, delete | 4 |
| 3 | Linear search | 2 |
| 4 | Bubble sort | 2 |
| 5 | Selection sort | 2 |
| 6 | Priority, count, category functions | 2 |
| 7 | Testing | 2 |
| 8 | Complexity analysis | 1 |
| 9 | Interactive menu | - |
| All | Code organization and explanation | 1 |

**How to use:** run the cells from top to bottom (*Run All*). The last cell starts the interactive menu.

## 1. Linked List Implementation
A linked list is a chain of **nodes**. Each node stores one contact and a `next` pointer to the following node. `head` points to the first node, and the last node's `next` is `None`.

```
head -> [Police] -> [Hospital] -> [Family] -> None
```

The next two cells define the `Node` class and the `LinkedList` class. The `LinkedList` class holds every operation (add, display, update, delete, search, sorts, priority, count, category), so the later sections only *use* it.

In [1]:
class Node:
    """One contact stored in the list. 'next' points to the next node."""

    def __init__(self, contact_id, name, phone, category, priority):
        self.contact_id = contact_id
        self.name = name
        self.phone = phone
        self.category = category
        self.priority = priority
        self.next = None  # last node's next stays None

In [2]:
class LinkedList:
    """Singly linked list of emergency contacts."""

    def __init__(self):
        self.head = None  # empty list

    # ---------- display helpers ----------
    @staticmethod
    def _header():
        print(f"\n{'ID':<8}{'Name':<20}{'Phone':<16}{'Category':<14}{'Priority':<8}")
        print("-" * 66)

    @staticmethod
    def _row(node):
        print(f"{node.contact_id:<8}{node.name:<20}{node.phone:<16}"
              f"{node.category:<14}{node.priority:<8}")

    def _find_by_id(self, contact_id):
        """Return the node with this ID, or None."""
        current = self.head
        while current:
            if current.contact_id.lower() == contact_id.strip().lower():
                return current
            current = current.next
        return None

    # =================================================================
    # SECTION 2: ADD, DISPLAY, UPDATE, DELETE
    # =================================================================
    def add(self, contact_id, name, phone, category, priority):
        """Add a contact at the end of the list. Returns False if ID exists."""
        if self._find_by_id(contact_id):
            return False
        new_node = Node(contact_id.upper(), name, phone, category, priority)
        if self.head is None:          # empty list: new node becomes head
            self.head = new_node
        else:                          # otherwise walk to the last node
            current = self.head
            while current.next:
                current = current.next
            current.next = new_node
        return True

    def display(self):
        """Print every contact from head to tail."""
        if self.head is None:
            print("No contacts to show.")
            return
        self._header()
        current = self.head
        while current:
            self._row(current)
            current = current.next

    def update(self, contact_id, name=None, phone=None, category=None, priority=None):
        """Change the given fields of a contact. None means keep the old value."""
        node = self._find_by_id(contact_id)
        if node is None:
            return False
        if name:
            node.name = name
        if phone:
            node.phone = phone
        if category:
            node.category = category
        if priority is not None:
            node.priority = priority
        return True

    def delete(self, contact_id):
        """Remove a contact by re-linking the previous node past it."""
        current, previous = self.head, None
        while current:
            if current.contact_id.lower() == contact_id.strip().lower():
                if previous is None:           # deleting the head
                    self.head = current.next
                else:                          # deleting a middle/last node
                    previous.next = current.next
                return True
            previous, current = current, current.next
        return False

    # =================================================================
    # SECTION 3: LINEAR SEARCH
    # =================================================================
    def linear_search(self, key):
        """Check nodes one by one from the head.
        A node matches if its ID equals the key or its name contains the key.
        Returns a list of matching nodes (empty list if none)."""
        key = key.strip().lower()
        matches = []
        current = self.head
        while current:                                   # visit each node
            if current.contact_id.lower() == key or key in current.name.lower():
                matches.append(current)
            current = current.next
        return matches

    def show_search(self, key):
        results = self.linear_search(key)
        if not results:
            print("No matching contact found.")
            return
        self._header()
        for node in results:
            self._row(node)

    # =================================================================
    # SECTION 4: BUBBLE SORT (by priority, 1 first)
    # =================================================================
    @staticmethod
    def _swap_data(a, b):
        """Swap the contents of two nodes (links stay untouched)."""
        a.contact_id, b.contact_id = b.contact_id, a.contact_id
        a.name, b.name = b.name, a.name
        a.phone, b.phone = b.phone, a.phone
        a.category, b.category = b.category, a.category
        a.priority, b.priority = b.priority, a.priority

    def bubble_sort(self):
        """Repeatedly compare neighbours and swap if out of order.
        After each pass the largest priority 'bubbles' to the end.
        Stops early if a full pass makes no swap."""
        if self.head is None:
            return
        swapped = True
        end = None                      # nodes from 'end' onward are already sorted
        while swapped:
            swapped = False
            current = self.head
            while current.next is not end:
                if current.priority > current.next.priority:
                    self._swap_data(current, current.next)
                    swapped = True
                current = current.next
            end = current               # last node of this pass is now in place

    # =================================================================
    # SECTION 5: SELECTION SORT (alphabetically by name)
    # =================================================================
    def selection_sort(self):
        """For each position, find the smallest name in the unsorted part
        and swap it into that position."""
        current = self.head
        while current:
            smallest = current
            scan = current.next
            while scan:                 # find the minimum in the rest of the list
                if scan.name.lower() < smallest.name.lower():
                    smallest = scan
                scan = scan.next
            if smallest is not current:
                self._swap_data(current, smallest)
            current = current.next

    # =================================================================
    # SECTION 6: PRIORITY / COUNT / CATEGORY FUNCTIONS
    # =================================================================
    def highest_priority(self):
        """Return all contacts sharing the best (lowest-number) priority."""
        if self.head is None:
            return []
        best = self.head.priority
        current = self.head
        while current:
            if current.priority < best:
                best = current.priority
            current = current.next
        result, current = [], self.head
        while current:
            if current.priority == best:
                result.append(current)
            current = current.next
        return result

    def count(self):
        """Count nodes by walking the list."""
        total, current = 0, self.head
        while current:
            total += 1
            current = current.next
        return total

    def by_category(self, category):
        """Return all contacts in the given category (case-insensitive)."""
        result, current = [], self.head
        while current:
            if current.category.lower() == category.strip().lower():
                result.append(current)
            current = current.next
        return result

    def categories(self):
        seen, current = [], self.head
        while current:
            if current.category not in seen:
                seen.append(current.category)
            current = current.next
        return seen

    # ---------- helpers used by tests ----------
    def to_names(self):
        names, current = [], self.head
        while current:
            names.append(current.name)
            current = current.next
        return names

    def to_priorities(self):
        values, current = [], self.head
        while current:
            values.append(current.priority)
            current = current.next
        return values

### Starting data (12 records)
The project requires at least 10 records. The first three are the examples from the assignment.

In [3]:
def build_sample_list():
    """The three example contacts from the assignment (used by the tests)."""
    lst = LinkedList()
    lst.add("E001", "Police", "999", "Emergency", 1)
    lst.add("E002", "Hospital", "911", "Medical", 1)
    lst.add("E003", "Family", "0712345678", "Family", 2)
    return lst


def build_full_data():
    """Starting data for the program: 12 records (requirement: at least 10).
    The first three are the examples from the assignment."""
    lst = LinkedList()
    records = [
        ("E001", "Police",          "999",        "Emergency", 1),
        ("E002", "Hospital",        "911",        "Medical",   1),
        ("E003", "Family",          "0712345678", "Family",    2),
        ("E004", "Fire Brigade",    "112",        "Emergency", 1),
        ("E005", "Ambulance",       "0722000001", "Medical",   1),
        ("E006", "Mother",          "0733456789", "Family",    2),
        ("E007", "Father",          "0744567890", "Family",    2),
        ("E008", "Family Doctor",   "0755678901", "Medical",   2),
        ("E009", "Neighbour",       "0766789012", "Friend",    3),
        ("E010", "Best Friend",     "0777890123", "Friend",    3),
        ("E011", "Security Guard",  "0788901234", "Emergency", 2),
        ("E012", "Pharmacy",        "0799012345", "Medical",   3),
    ]
    for record in records:
        lst.add(*record)
    return lst

In [4]:
contacts = build_full_data()
print('Records loaded:', contacts.count())

Records loaded: 12


## 2. Add, Display, Update and Delete
- **add:** walks to the last node and links the new node there. Duplicate IDs are rejected.
- **display:** visits every node from head to tail.
- **update:** finds the node by ID and changes only the fields that are given.
- **delete:** re-links the previous node past the deleted one. It handles first, middle and last nodes.

In [5]:
print("DISPLAY: all contacts")
contacts.display()

DISPLAY: all contacts

ID      Name                Phone           Category      Priority
------------------------------------------------------------------
E001    Police              999             Emergency     1       
E002    Hospital            911             Medical       1       
E003    Family              0712345678      Family        2       
E004    Fire Brigade        112             Emergency     1       
E005    Ambulance           0722000001      Medical       1       
E006    Mother              0733456789      Family        2       
E007    Father              0744567890      Family        2       
E008    Family Doctor       0755678901      Medical       2       
E009    Neighbour           0766789012      Friend        3       
E010    Best Friend         0777890123      Friend        3       
E011    Security Guard      0788901234      Emergency     2       
E012    Pharmacy            0799012345      Medical       3       


In [6]:
print("ADD: new contact E013")
print("Added:", contacts.add("E013", "Dr. Kamau", "0711222333", "Medical", 2))
print("ADD: duplicate ID E013 is rejected ->", contacts.add("E013", "Other", "0700000000", "Friend", 3))
contacts.display()

ADD: new contact E013
Added: True
ADD: duplicate ID E013 is rejected -> False

ID      Name                Phone           Category      Priority
------------------------------------------------------------------
E001    Police              999             Emergency     1       
E002    Hospital            911             Medical       1       
E003    Family              0712345678      Family        2       
E004    Fire Brigade        112             Emergency     1       
E005    Ambulance           0722000001      Medical       1       
E006    Mother              0733456789      Family        2       
E007    Father              0744567890      Family        2       
E008    Family Doctor       0755678901      Medical       2       
E009    Neighbour           0766789012      Friend        3       
E010    Best Friend         0777890123      Friend        3       
E011    Security Guard      0788901234      Emergency     2       
E012    Pharmacy            0799012345      Medica

In [7]:
print("UPDATE: change E013 to priority 1 and a new phone number")
print("Updated:", contacts.update("E013", phone="0799888777", priority=1))
print("UPDATE: unknown ID E999 ->", contacts.update("E999", name="Nobody"))
contacts.display()

UPDATE: change E013 to priority 1 and a new phone number
Updated: True
UPDATE: unknown ID E999 -> False

ID      Name                Phone           Category      Priority
------------------------------------------------------------------
E001    Police              999             Emergency     1       
E002    Hospital            911             Medical       1       
E003    Family              0712345678      Family        2       
E004    Fire Brigade        112             Emergency     1       
E005    Ambulance           0722000001      Medical       1       
E006    Mother              0733456789      Family        2       
E007    Father              0744567890      Family        2       
E008    Family Doctor       0755678901      Medical       2       
E009    Neighbour           0766789012      Friend        3       
E010    Best Friend         0777890123      Friend        3       
E011    Security Guard      0788901234      Emergency     2       
E012    Pharmacy        

In [8]:
print("DELETE: remove E013 (the last node) and E001 (the first node)")
print("Deleted E013:", contacts.delete("E013"))
print("Deleted E001:", contacts.delete("E001"))
print("DELETE: unknown ID E999 ->", contacts.delete("E999"))
contacts.display()

DELETE: remove E013 (the last node) and E001 (the first node)
Deleted E013: True
Deleted E001: True
DELETE: unknown ID E999 -> False

ID      Name                Phone           Category      Priority
------------------------------------------------------------------
E002    Hospital            911             Medical       1       
E003    Family              0712345678      Family        2       
E004    Fire Brigade        112             Emergency     1       
E005    Ambulance           0722000001      Medical       1       
E006    Mother              0733456789      Family        2       
E007    Father              0744567890      Family        2       
E008    Family Doctor       0755678901      Medical       2       
E009    Neighbour           0766789012      Friend        3       
E010    Best Friend         0777890123      Friend        3       
E011    Security Guard      0788901234      Emergency     2       
E012    Pharmacy            0799012345      Medical       3   

## 3. Linear Search
Linear search checks the nodes **one by one** from the head. A node matches if its ID equals the search text, or its name contains the search text.

In [9]:
for key in ["E003", "doctor", "Zebra"]:
    print(f"Searching for '{key}':")
    contacts.show_search(key)
    print()

Searching for 'E003':

ID      Name                Phone           Category      Priority
------------------------------------------------------------------
E003    Family              0712345678      Family        2       

Searching for 'doctor':

ID      Name                Phone           Category      Priority
------------------------------------------------------------------
E008    Family Doctor       0755678901      Medical       2       

Searching for 'Zebra':
No matching contact found.



## 4. Bubble Sort (by priority)
Bubble sort repeatedly compares **neighbouring** nodes and swaps them if they are out of order. After each pass the largest priority value "bubbles" to the end. It stops early if a full pass makes no swaps. Priority 1 comes first.

In [10]:
print("Before:", contacts.to_priorities())
contacts.bubble_sort()
print("After bubble sort:", contacts.to_priorities())
contacts.display()

Before: [1, 2, 1, 1, 2, 2, 2, 3, 3, 2, 3]
After bubble sort: [1, 1, 1, 2, 2, 2, 2, 2, 3, 3, 3]

ID      Name                Phone           Category      Priority
------------------------------------------------------------------
E002    Hospital            911             Medical       1       
E004    Fire Brigade        112             Emergency     1       
E005    Ambulance           0722000001      Medical       1       
E003    Family              0712345678      Family        2       
E006    Mother              0733456789      Family        2       
E007    Father              0744567890      Family        2       
E008    Family Doctor       0755678901      Medical       2       
E011    Security Guard      0788901234      Emergency     2       
E009    Neighbour           0766789012      Friend        3       
E010    Best Friend         0777890123      Friend        3       
E012    Pharmacy            0799012345      Medical       3       


## 5. Selection Sort (alphabetically by name)
Selection sort goes through the list position by position. For each position it **finds the smallest name** in the unsorted part and swaps it into place.

In [11]:
print("Before:", contacts.to_names())
contacts.selection_sort()
print("After selection sort:", contacts.to_names())
contacts.display()

Before: ['Hospital', 'Fire Brigade', 'Ambulance', 'Family', 'Mother', 'Father', 'Family Doctor', 'Security Guard', 'Neighbour', 'Best Friend', 'Pharmacy']
After selection sort: ['Ambulance', 'Best Friend', 'Family', 'Family Doctor', 'Father', 'Fire Brigade', 'Hospital', 'Mother', 'Neighbour', 'Pharmacy', 'Security Guard']

ID      Name                Phone           Category      Priority
------------------------------------------------------------------
E005    Ambulance           0722000001      Medical       1       
E010    Best Friend         0777890123      Friend        3       
E003    Family              0712345678      Family        2       
E008    Family Doctor       0755678901      Medical       2       
E007    Father              0744567890      Family        2       
E004    Fire Brigade        112             Emergency     1       
E002    Hospital            911             Medical       1       
E006    Mother              0733456789      Family        2       
E009 

## 6. Priority, Count and Category Functions
- **highest_priority:** finds the lowest priority number, then returns every contact that has it.
- **count:** counts the nodes.
- **by_category:** returns the contacts in one category.

In [12]:
print("Highest-priority contact(s):")
for node in contacts.highest_priority():
    print("  ", node.name, "(priority", node.priority, ")")

print("\nTotal contacts:", contacts.count())

print("\nCategories:", contacts.categories())
print("Contacts in 'Medical':")
for node in contacts.by_category("Medical"):
    print("  ", node.name)

Highest-priority contact(s):
   Ambulance (priority 1 )
   Fire Brigade (priority 1 )
   Hospital (priority 1 )

Total contacts: 11

Categories: ['Medical', 'Friend', 'Family', 'Emergency']
Contacts in 'Medical':
   Ambulance
   Family Doctor
   Hospital
   Pharmacy


## 7. Testing
Each test checks a result against the expected value and prints **PASS** or **FAIL**. The tests cover add, duplicate IDs, update, linear search, both sorts, highest priority, category, delete (first, middle, missing) and empty-list cases.

In [13]:
def check(description, actual, expected):
    status = "PASS" if actual == expected else "FAIL"
    print(f"[{status}] {description}: got {actual}, expected {expected}")
    return actual == expected

In [14]:
def run_tests():
    print("\n===== RUNNING TESTS =====")
    results = []

    # Requirement: at least 10 records
    full = build_full_data()
    results.append(check("Program starts with at least 10 records", full.count() >= 10, True))
    full.bubble_sort()
    results.append(check("Bubble sort on 12 records",
                         full.to_priorities() == sorted(full.to_priorities()), True))
    full.selection_sort()
    results.append(check("Selection sort on 12 records",
                         [n.lower() for n in full.to_names()] ==
                         sorted(n.lower() for n in full.to_names()), True))
    results.append(check("Count of 'Medical' contacts in 12 records",
                         len(build_full_data().by_category("Medical")), 4))

    lst = build_sample_list()
    results.append(check("Count after adding 3 contacts", lst.count(), 3))
    results.append(check("Duplicate ID rejected", lst.add("E001", "X", "1", "Y", 1), False))

    lst.add("E004", "Fire Dept", "112", "Emergency", 3)
    results.append(check("Count after adding E004", lst.count(), 4))
    results.append(check("Order after adds", lst.to_names(),
                         ["Police", "Hospital", "Family", "Fire Dept"]))

    # Update
    results.append(check("Update existing contact", lst.update("E004", priority=1), True))
    results.append(check("Update missing contact", lst.update("E999", name="Z"), False))
    results.append(check("Priority changed to 1", lst.to_priorities(), [1, 1, 2, 1]))

    # Linear search
    results.append(check("Search by ID", [n.name for n in lst.linear_search("E003")], ["Family"]))
    results.append(check("Search partial name", [n.name for n in lst.linear_search("fire")], ["Fire Dept"]))
    results.append(check("Search not found", lst.linear_search("Zebra"), []))

    # Bubble sort
    lst.bubble_sort()
    results.append(check("Bubble sort by priority", lst.to_priorities(), [1, 1, 1, 2]))

    # Selection sort
    lst.selection_sort()
    results.append(check("Selection sort by name", lst.to_names(),
                         ["Family", "Fire Dept", "Hospital", "Police"]))

    # Highest priority, category
    results.append(check("Highest priority contacts",
                         sorted(n.name for n in lst.highest_priority()),
                         ["Fire Dept", "Hospital", "Police"]))
    results.append(check("Category 'Emergency'",
                         sorted(n.name for n in lst.by_category("emergency")),
                         ["Fire Dept", "Police"]))

    # Delete: head, middle, missing
    results.append(check("Delete head", lst.delete("E003"), True))
    results.append(check("Delete middle/last", lst.delete("E001"), True))
    results.append(check("Delete missing", lst.delete("E999"), False))
    results.append(check("Count after deletes", lst.count(), 2))

    # Empty list edge cases
    empty = LinkedList()
    empty.bubble_sort()
    empty.selection_sort()
    results.append(check("Empty list count", empty.count(), 0))
    results.append(check("Empty list highest priority", empty.highest_priority(), []))

    print(f"\n{sum(results)}/{len(results)} tests passed.")
    return all(results)

In [15]:
all_passed = run_tests()
print('\nAll tests passed:', all_passed)


===== RUNNING TESTS =====
[PASS] Program starts with at least 10 records: got True, expected True
[PASS] Bubble sort on 12 records: got True, expected True
[PASS] Selection sort on 12 records: got True, expected True
[PASS] Count of 'Medical' contacts in 12 records: got 4, expected 4
[PASS] Count after adding 3 contacts: got 3, expected 3
[PASS] Duplicate ID rejected: got False, expected False
[PASS] Count after adding E004: got 4, expected 4
[PASS] Order after adds: got ['Police', 'Hospital', 'Family', 'Fire Dept'], expected ['Police', 'Hospital', 'Family', 'Fire Dept']
[PASS] Update existing contact: got True, expected True
[PASS] Update missing contact: got False, expected False
[PASS] Priority changed to 1: got [1, 1, 2, 1], expected [1, 1, 2, 1]
[PASS] Search by ID: got ['Family'], expected ['Family']
[PASS] Search partial name: got ['Fire Dept'], expected ['Fire Dept']
[PASS] Search not found: got [], expected []
[PASS] Bubble sort by priority: got [1, 1, 1, 2], expected [1, 1, 

## 8. Complexity Analysis
Here *n* is the number of contacts.

| Operation | Time | Space | Explanation |
|---|---|---|---|
| Add (at the end) | O(n) | O(1) | walk to the last node (O(1) if a tail pointer were kept) |
| Display | O(n) | O(1) | visit every node once |
| Update / Delete | O(n) | O(1) | must search for the node first |
| Linear Search | O(n) | O(1) | best case O(1), worst case O(n) |
| Bubble Sort | O(n²) | O(1) | n passes, each comparing neighbours (best case O(n) with early exit) |
| Selection Sort | O(n²) | O(1) | always scans the unsorted part to find the minimum |
| Highest priority | O(n) | O(1) | one pass through the list |
| Count | O(n) | O(1) | one pass (O(1) if a size counter were stored) |
| Display by category | O(n) | O(1) | one pass with a filter |

Both sorts swap the **data inside nodes**, not the nodes themselves, so they use no extra memory. They are both O(n²), but selection sort makes fewer swaps (at most n-1).

The same table is printed by the program itself (menu option 12):

In [16]:
def show_complexity():
    """Print the time complexity of the main operations (n = number of contacts)."""
    print("\n--- Time Complexity of Main Operations (n = number of contacts) ---")
    rows = [
        ("Add (go to the end)", "O(n)",   "Walk to the last node"),
        ("Display",             "O(n)",   "Visit every node once"),
        ("Linear Search",       "O(n)",   "Worst case checks every node"),
        ("Update / Delete",     "O(n)",   "Must search for the node first"),
        ("Bubble Sort",         "O(n^2)", "Compares neighbours, n passes"),
        ("Selection Sort",      "O(n^2)", "Scans the unsorted part each time"),
        ("Highest priority",    "O(n)",   "One pass through the list"),
        ("Count",               "O(n)",   "One pass through the list"),
        ("Display by category", "O(n)",   "One pass with a filter"),
    ]
    print(f"{'Operation':<22}{'Time':<9}Reason")
    print("-" * 60)
    for operation, time, reason in rows:
        print(f"{operation:<22}{time:<9}{reason}")

In [17]:
show_complexity()


--- Time Complexity of Main Operations (n = number of contacts) ---
Operation             Time     Reason
------------------------------------------------------------
Add (go to the end)   O(n)     Walk to the last node
Display               O(n)     Visit every node once
Linear Search         O(n)     Worst case checks every node
Update / Delete       O(n)     Must search for the node first
Bubble Sort           O(n^2)   Compares neighbours, n passes
Selection Sort        O(n^2)   Scans the unsorted part each time
Highest priority      O(n)     One pass through the list
Count                 O(n)     One pass through the list
Display by category   O(n)     One pass with a filter


### Measured running time
This cell checks the analysis by timing each sort on lists of different sizes. When the size doubles, an O(n²) sort should take about **4 times** longer.

In [18]:
import time, random

def timed(sort_name, size):
    lst = LinkedList()
    for i in range(size):
        lst.add(f"E{i}", f"Name{random.randint(0, 10**6)}", "0700000000", "Test", random.randint(1, 1000))
    start = time.perf_counter()
    getattr(lst, sort_name)()
    return time.perf_counter() - start

print(f"{'Size':>6}  {'Bubble (s)':>12}  {'Selection (s)':>14}")
for size in [200, 400, 800]:
    print(f"{size:>6}  {timed('bubble_sort', size):>12.4f}  {timed('selection_sort', size):>14.4f}")

  Size    Bubble (s)   Selection (s)
   200        0.0026          0.0016
   400        0.0108          0.0067


   800        0.0416          0.0277


## 9. Interactive Menu
The menu lets the user choose add, display, search, update, delete, sort, highest priority, count, category, tests, complexity or exit. Type your answer in the **input box** that appears at the top of the editor (or under the cell), then press **Enter**. Enter `0` to exit.

In [19]:
def read_nonempty(prompt):
    while True:
        value = input(prompt).strip()
        if value:
            return value
        print("This field cannot be empty.")


def read_phone(prompt, allow_blank=False):
    while True:
        value = input(prompt).strip()
        if not value and allow_blank:
            return None
        if value and value.replace("+", "", 1).isdigit():
            return value
        print("Phone number must contain digits only (a leading + is allowed).")


def read_priority(prompt, allow_blank=False):
    while True:
        value = input(prompt).strip()
        if not value and allow_blank:
            return None
        if value.isdigit() and int(value) >= 1:
            return int(value)
        print("Priority must be a whole number of 1 or more (1 = highest).")


def menu_add(lst):
    print("\n--- Add a Contact ---")
    contact_id = read_nonempty("Contact ID (e.g. E004): ").upper()
    name = read_nonempty("Name: ")
    phone = read_phone("Phone number: ")
    category = read_nonempty("Category: ").title()
    priority = read_priority("Priority (1 = highest): ")
    if lst.add(contact_id, name, phone, category, priority):
        print(f"Contact '{name}' added.")
    else:
        print("A contact with that ID already exists.")


def menu_update(lst):
    print("\n--- Update Contact Information ---")
    contact_id = read_nonempty("Contact ID to update: ")
    if lst._find_by_id(contact_id) is None:
        print("Contact not found.")
        return
    print("Leave a field blank to keep its current value.")
    name = input("New name: ").strip() or None
    phone = read_phone("New phone: ", allow_blank=True)
    category = input("New category: ").strip().title() or None
    priority = read_priority("New priority: ", allow_blank=True)
    lst.update(contact_id, name, phone, category, priority)
    print("Contact updated.")


def menu_delete(lst):
    print("\n--- Delete a Contact ---")
    contact_id = read_nonempty("Contact ID to delete: ")
    node = lst._find_by_id(contact_id)
    if node is None:
        print("Contact not found.")
    elif input(f"Delete '{node.name}'? (y/n): ").strip().lower() == "y":
        lst.delete(contact_id)
        print("Contact deleted.")
    else:
        print("Deletion cancelled.")


def menu_highest(lst):
    print("\n--- Highest-Priority Contact(s) ---")
    nodes = lst.highest_priority()
    if not nodes:
        print("No contacts available.")
        return
    lst._header()
    for n in nodes:
        lst._row(n)


def menu_category(lst):
    print("\n--- Display Contacts by Category ---")
    if lst.head is None:
        print("No contacts available.")
        return
    print("Available categories:", ", ".join(lst.categories()))
    nodes = lst.by_category(read_nonempty("Enter a category: "))
    if not nodes:
        print("No contacts in that category.")
        return
    lst._header()
    for n in nodes:
        lst._row(n)


def show_complexity():
    """Print the time complexity of the main operations (n = number of contacts)."""
    print("\n--- Time Complexity of Main Operations (n = number of contacts) ---")
    rows = [
        ("Add (go to the end)", "O(n)",   "Walk to the last node"),
        ("Display",             "O(n)",   "Visit every node once"),
        ("Linear Search",       "O(n)",   "Worst case checks every node"),
        ("Update / Delete",     "O(n)",   "Must search for the node first"),
        ("Bubble Sort",         "O(n^2)", "Compares neighbours, n passes"),
        ("Selection Sort",      "O(n^2)", "Scans the unsorted part each time"),
        ("Highest priority",    "O(n)",   "One pass through the list"),
        ("Count",               "O(n)",   "One pass through the list"),
        ("Display by category", "O(n)",   "One pass with a filter"),
    ]
    print(f"{'Operation':<22}{'Time':<9}Reason")
    print("-" * 60)
    for operation, time, reason in rows:
        print(f"{operation:<22}{time:<9}{reason}")


MENU = """
===== EMERGENCY CONTACTS MANAGER (Linked List) =====
 1. Add a contact
 2. Display contacts
 3. Search for a contact            (Linear Search)
 4. Update contact information
 5. Delete a contact
 6. Sort contacts by priority       (Bubble Sort)
 7. Sort contacts alphabetically    (Selection Sort)
 8. Find the highest-priority contact
 9. Count contacts
10. Display contacts by category
11. Run tests
12. Show time complexity
 0. Exit
"""


def main():
    contacts = build_full_data()        # start with 12 records (at least 10 required)
    while True:
        print(MENU)
        choice = input("Choose an option (0-12): ").strip()
        if choice == "1":
            menu_add(contacts)
        elif choice == "2":
            print("\n--- All Contacts ---")
            contacts.display()
        elif choice == "3":
            print("\n--- Search for a Contact ---")
            contacts.show_search(read_nonempty("Enter ID or name: "))
        elif choice == "4":
            menu_update(contacts)
        elif choice == "5":
            menu_delete(contacts)
        elif choice == "6":
            contacts.bubble_sort()
            print("\n--- Sorted by Priority (Bubble Sort) ---")
            contacts.display()
        elif choice == "7":
            contacts.selection_sort()
            print("\n--- Sorted by Name (Selection Sort) ---")
            contacts.display()
        elif choice == "8":
            menu_highest(contacts)
        elif choice == "9":
            print(f"\nTotal contacts: {contacts.count()}")
        elif choice == "10":
            menu_category(contacts)
        elif choice == "11":
            run_tests()
        elif choice == "12":
            show_complexity()
        elif choice == "0":
            print("Goodbye! Stay safe.")
            break
        else:
            print("Invalid choice. Please enter a number from 0 to 12.")

In [ ]:
main()

## Conclusion
The program meets the project requirements:
- **Data structure:** a singly linked list built from `Node` and `LinkedList`.
- **Algorithms:** linear search, bubble sort and selection sort, written by hand.
- **Operations:** add, display, search, update, delete, sort, highest priority, count, category and exit, through a menu.
- **Data:** 12 starting records (at least 10 required).
- **Testing:** every major operation is checked with expected results.
- **Complexity:** search, update, delete, display, count and category are O(n). Bubble sort and selection sort are O(n²).